# Model 8 inference from model.pkl
Attach the fitted model.pkl and competition inputs, then run all cells.

In [ ]:
from pathlib import Path
import pickle,time
started=time.perf_counter()
# Set MODEL_PATH to the exact file path copied from Kaggle's Input pane if needed.
MODEL_PATH = None
from zipfile import ZipFile, is_zipfile
from contextlib import ExitStack
if MODEL_PATH:
    paths=[Path(MODEL_PATH)]
elif Path('/kaggle/input').is_dir():
    roots=[Path('/kaggle/input'),Path('/kaggle/working')]
    files=[p for root in roots if root.is_dir() for p in root.rglob('*')
           if p.suffix.lower() in ('.pkl','.zip') and p.is_file()]
    paths=[p for p in files if p.name.lower()=='model.pkl']
    if not paths:
        paths=[p for p in files if p.suffix.lower()=='.pkl' and p.stem.lower().startswith('model')]
    if not paths:
        for p in files:
            if p.suffix.lower()=='.zip' and is_zipfile(p):
                with ZipFile(p) as archive:
                    if any(Path(k).name.lower()=='model.pkl' for k in archive.namelist()):
                        paths.append(p)
    if not paths:
        print('Mounted Kaggle inputs:', [str(p) for p in roots[0].glob('*')])
        print('Pickles and ZIP files found:', [str(p) for p in files][:40])
        raise FileNotFoundError('Add your private weights dataset under Input, or set MODEL_PATH to your uploaded Model 8 pickle.')
else:
    root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Models/Emad').is_dir())
    paths=[root/'Models/Emad/model3e_plus_cache/model.pkl']
if len(paths)!=1: raise ValueError(f'Set MODEL_PATH to the desired weights file. Candidates: {paths}')
print('Loading weights:',paths[0],flush=True)
with ExitStack() as stack:
    if paths[0].suffix.lower()=='.zip':
        archive=stack.enter_context(ZipFile(paths[0]))
        keys=[k for k in archive.namelist() if Path(k).name.lower()=='model.pkl']
        if len(keys)!=1: raise ValueError(f'Expected one model.pkl in {paths[0]}, found {keys}')
        stream=stack.enter_context(archive.open(keys[0]))
    else:
        stream=stack.enter_context(paths[0].open('rb'))
    package=pickle.load(stream)
assert package['format']=='model8-v1'
exec(package['feature_code'])
bundle=WORK/'model8_pickle_runtime'
bundle.mkdir(parents=True,exist_ok=True)
(bundle/'public_sift_pairs.npz').write_bytes(package.pop('public_geometry'))
result=score_artifact(bundle,package['model'])
out=WORK/'submission_model8.csv'
result.to_csv(out,index=False)
print(out,len(result),'rows',round(time.perf_counter()-started,1),'seconds')
